# Load Data


In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

END_DATE = datetime.today().strftime("%Y-%m-%d")
# Начало – 1 января 2017 (для крипты и акций)
START_DATE = "2017-01-01"

print(f"Период загрузки: {START_DATE} – {END_DATE}")

# Импорт функций из нашего модуля
from funcs.load_data import fetch_moex_history, fetch_yahoo_daily, get_russian_stock_data

Период загрузки: 2017-01-01 – 2026-05-01


## Загрузка данных криптовалют

In [3]:
# Словарь: тикер Yahoo -> имя файла
crypto = {
    "BTC-USD": "btc_daily.csv",
    "ETH-USD": "eth_daily.csv",
}

for ticker, filename in crypto.items():
    filepath = os.path.join("data", filename)
    if os.path.exists(filepath):
        print(f"Файл {filepath} уже существует, пропускаем загрузку.")
        continue
    df = fetch_yahoo_daily(ticker, START_DATE, END_DATE)
    if not df.empty:
        df.to_csv(filepath)
        print(f"Сохранён: {filepath}")

Файл data\btc_daily.csv уже существует, пропускаем загрузку.
Файл data\eth_daily.csv уже существует, пропускаем загрузку.


In [4]:
df_btc = pd.read_csv(f'data/{crypto["BTC-USD"]}')
df_btc.shape, df_btc.columns, df_btc['date'].min(), df_btc['date'].max()

((3407, 2),
 Index(['date', 'close'], dtype='object'),
 '2017-01-01',
 '2026-04-30')

In [5]:
df_eth = pd.read_csv(f'data/{crypto["ETH-USD"]}')
df_eth.shape, df_eth.columns, df_eth['date'].min(), df_eth['date'].max()

((3095, 2),
 Index(['date', 'close'], dtype='object'),
 '2017-11-09',
 '2026-04-30')

## Загрузка данных акций 

In [6]:
sber_file = os.path.join("data", "sber_daily.csv")
if not os.path.exists(sber_file):
    df_sber = fetch_moex_history("SBER", START_DATE, END_DATE)
    if not df_sber.empty:
        df_sber.to_csv(sber_file)
        print(f"Сохранён: {sber_file}")
else:
    print(f"Файл {sber_file} уже существует.")

Файл data\sber_daily.csv уже существует.


In [7]:
yandex_file = os.path.join("data", "yandex_daily.csv")
if not os.path.exists(yandex_file):
    # Объединяем историю YNDX (до делистинга) и YDEX (после 2024)
    df_yandex = get_russian_stock_data("YNDX", "YDEX", START_DATE, END_DATE)
    if not df_yandex.empty:
        df_yandex.to_csv(yandex_file)
        print(f"Сохранён: {yandex_file}")
else:
    print(f"Файл {yandex_file} уже существует.")

Файл data\yandex_daily.csv уже существует.


In [9]:
df_sber = pd.read_csv(f'data/sber_daily.csv')
df_sber.shape, df_sber.columns, df_sber['date'].min(), df_sber['date'].max()

((2361, 2),
 Index(['date', 'close'], dtype='object'),
 '2017-01-03',
 '2026-04-30')

In [10]:
df_yndx = pd.read_csv(f'data/yandex_daily.csv')
df_yndx.shape, df_yndx.columns, df_yndx['date'].min(), df_yndx['date'].max()

((2361, 2),
 Index(['date', 'close'], dtype='object'),
 '2017-01-03',
 '2026-04-30')

## Тест данных

In [13]:
# Загружаем все CSV для проверки
files = {
    "SBER": "sber_daily.csv",
    "YNDX": "yandex_daily.csv",
    "BTC": "btc_daily.csv",
    "ETH": "eth_daily.csv",
}

data = {}
for name, fname in files.items():
    path = os.path.join("data", fname)
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        # Убедимся, что индекс DatetimeIndex
        df.index = pd.to_datetime(df.index)
        data[name] = df["close"]
        print(f"{name:>5s}: {len(df)} записей, {df.index.min().date()} – {df.index.max().date()}")
    else:
        print(f"{name:>5s}: ФАЙЛ НЕ НАЙДЕН!")

# Объединим в одну таблицу для визуализации
if data:
    all_close = pd.DataFrame(data)
    print("\nОбщая таблица (последние 5 строк):")
    display(all_close.tail())
else:
    print("Нет данных для отображения.")

 SBER: 2361 записей, 2017-01-03 – 2026-04-30
 YNDX: 2361 записей, 2017-01-03 – 2026-04-30
  BTC: 3407 записей, 2017-01-01 – 2026-04-30
  ETH: 3095 записей, 2017-11-09 – 2026-04-30

Общая таблица (последние 5 строк):


,SBER,YNDX,BTC,ETH
date,,,,
2026-04-26,NaN,NaN,78657.539062,2369.729004
2026-04-27,321.99,4151.0,77366.625000,2303.063965
2026-04-28,319.82,4056.5,76350.671875,2289.419678
2026-04-29,320.09,4013.5,75776.132812,2253.415771
2026-04-30,319.81,4060.0,76304.320312,2256.251221


In [19]:
print("=== Проверка пропусков (NaN) в уже загруженных данных ===")
for col in all_close.columns:
    nulls = all_close[col].isnull().sum()
    print(f"{col}: {nulls} пропусков (из {all_close[col].notna().sum()} ненулевых записей)")

=== Проверка пропусков (NaN) в уже загруженных данных ===
SBER: 1064 пропусков (из 2343 ненулевых записей)
YNDX: 1094 пропусков (из 2313 ненулевых записей)
BTC: 0 пропусков (из 3407 ненулевых записей)
ETH: 312 пропусков (из 3095 ненулевых записей)


In [23]:
import holidays


# Российские праздники (официальные нерабочие дни)
ru_holidays = holidays.RU(years=range(2017, 2026))

def analyze_missing_dates(series, asset_name):
    """
    Для переданного ряда (с DatetimeIndex) определяет отсутствующие даты
    в диапазоне от min до max индекса.
    Возвращает DataFrame с колонками:
      - date
      - is_weekend (суббота/воскресенье)
      - is_ru_holiday (российский праздник)
      - is_trading_day (дата есть в индексе ряда)
    """
    start = series.index.min()
    end = series.index.max()
    full_range = pd.date_range(start, end, freq='D')

    df_cal = pd.DataFrame(index=full_range)
    df_cal.index.name = "date"
    df_cal["is_trading_day"] = df_cal.index.isin(series.index)
    df_cal["is_weekend"] = df_cal.index.dayofweek >= 5  # 5=Sat, 6=Sun
    df_cal["is_ru_holiday"] = df_cal.index.map(lambda d: d in ru_holidays)
    return df_cal

# Анализируем для акций и криптовалют
cal_data = {}
for name in all_close.columns:
    cal_data[name] = analyze_missing_dates(all_close[name], name)

# Пример для SBER: сколько дней нет в индексе
for name, cal_df in cal_data.items():
    missing = (~cal_df["is_trading_day"]).sum()
    weekends = cal_df[~cal_df["is_trading_day"]]["is_weekend"].sum()
    holidays_miss = cal_df[~cal_df["is_trading_day"]]["is_ru_holiday"].sum()
    print(f"\n{name}:")
    print(f"  Всего дней в диапазоне: {len(cal_df)}")
    print(f"  Отсутствуют в данных: {missing}")
    print(f"  Из них выходных (сб/вс): {weekends}")
    print(f"  Из них гос. праздников РФ: {holidays_miss}")
    # Проверим, есть ли пропущенные дни, которые не выходные и не праздники
    other = cal_df[~cal_df["is_trading_day"] & ~cal_df["is_weekend"] & ~cal_df["is_ru_holiday"]]
    if len(other) > 0:
        print(f"  ⚠️ Другие пропуски ({len(other)} дней):")
        print(other.head(10))


SBER:
  Всего дней в диапазоне: 3407
  Отсутствуют в данных: 0
  Из них выходных (сб/вс): 0
  Из них гос. праздников РФ: 0

YNDX:
  Всего дней в диапазоне: 3407
  Отсутствуют в данных: 0
  Из них выходных (сб/вс): 0
  Из них гос. праздников РФ: 0

BTC:
  Всего дней в диапазоне: 3407
  Отсутствуют в данных: 0
  Из них выходных (сб/вс): 0
  Из них гос. праздников РФ: 0

ETH:
  Всего дней в диапазоне: 3407
  Отсутствуют в данных: 0
  Из них выходных (сб/вс): 0
  Из них гос. праздников РФ: 0


In [18]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=list(all_close.columns))

for i, col in enumerate(all_close.columns):
    fig.add_trace(
        go.Scatter(x=all_close.index, y=all_close[col],
                   mode='lines', name=col, line=dict(width=1)),
        row=i+1, col=1
    )

fig.update_layout(height=900, title_text="Дневные цены закрытия (Plotly)")
fig.update_xaxes(rangeslider_visible=False)
fig.show()